# Cherry Tomato Vision Sorting — Colab Runner

## Goal
Run the repository pipeline against one video or a Drive directory, verify the bundled checkpoints and configuration, inspect the annotated outputs, and download all reports.

## Setup

Mount Drive and extract the release ZIP into a clean, predictable project directory.

The dependency cell intentionally shows full pip output. If it fails, stop and fix that cell before running inference.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile

from google.colab import drive

drive.mount('/content/drive')

DRIVE_ZIP_PATH = Path('/content/drive/MyDrive/cherry-tomato-vision-sorting.zip')
EXTRACT_ROOT = Path('/content')
PROJECT_ROOT = EXTRACT_ROOT / 'cherry-tomato-vision-sorting'

assert DRIVE_ZIP_PATH.is_file(), DRIVE_ZIP_PATH
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

with zipfile.ZipFile(DRIVE_ZIP_PATH) as archive:
    unsafe_member = next(
        (name for name in archive.namelist() if Path(name).is_absolute() or '..' in Path(name).parts),
        None,
    )
    if unsafe_member:
        raise ValueError(f'Unsafe ZIP member: {unsafe_member}')
    archive.extractall(EXTRACT_ROOT)

assert (PROJECT_ROOT / 'run_pipeline.py').is_file()
print('Project root:', PROJECT_ROOT)

In [ ]:
requirements_path = PROJECT_ROOT / 'requirements.txt'

print('Python:', sys.version)
print('Installing:', requirements_path)

# Keep pip tooling current, then install the repository's interpreter-aware pins.
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--upgrade',
    'pip',
    'setuptools',
    'wheel',
])
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--upgrade',
    '-r',
    str(requirements_path),
])

# Verify imports in a fresh interpreter so installation failures are visible here.
verification_code = """
import cv2
import lap
import numpy
import torch
import torchvision
import ultralytics
import yaml

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('ultralytics:', ultralytics.__version__)
print('opencv:', cv2.__version__)
print('numpy:', numpy.__version__)
print('CUDA available:', torch.cuda.is_available())
"""
print(subprocess.check_output([sys.executable, '-c', verification_code], text=True))

## Checks

Verify release filenames, hashes, model thresholds, size thresholds, and the temporal policy before running inference.

In [ ]:
import yaml

config = yaml.safe_load((PROJECT_ROOT / 'config/config.yaml').read_text(encoding='utf-8'))
manifest = PROJECT_ROOT / 'models/checksums.sha256'
expected_hashes = dict(line.split() for line in manifest.read_text().splitlines() if line.strip())

for relative_path, expected_digest in expected_hashes.items():
    model_path = PROJECT_ROOT / relative_path
    actual_digest = hashlib.sha256(model_path.read_bytes()).hexdigest()
    assert actual_digest == expected_digest, relative_path

assert config['detector']['weights_path'] == 'models/detector/yolo_tomato_detector.pt'
assert config['classifier']['weights_path'] == 'models/classifier/shufflenet_multitask.pt'
assert config['classification_policy']['mode'] == 'lower_zone_asymmetric_temporal_voting'
assert config['size_estimation']['metric'] == 'min_dimension'
assert 0 < config['size_estimation']['thresholds']['small_max_px'] < config['size_estimation']['thresholds']['medium_max_px']
print('Release configuration and checkpoints verified.')

## Run

Set the input and optional CLI overrides below. Every command-line option supported by `run_pipeline.py` is represented here. Leave an optional value as `None` or `False` to use the YAML default.

In [ ]:
VIDEO_SOURCE = Path('/content/drive/MyDrive/TomatoVideos')
OUTPUT_ROOT = PROJECT_ROOT / 'outputs/colab'
OUTPUT_VIDEO = None
RECURSIVE = False
STOP_ON_ERROR = False
DEVICE = None
MAX_FRAMES = None
DETECTOR_WEIGHTS = None
CLASSIFIER_WEIGHTS = None

SUPPORTED = {'.mp4', '.avi', '.mov', '.mkv', '.m4v', '.webm'}
if VIDEO_SOURCE.is_dir():
    iterator = VIDEO_SOURCE.rglob('*') if RECURSIVE else VIDEO_SOURCE.iterdir()
    videos = sorted(path for path in iterator if path.is_file() and path.suffix.lower() in SUPPORTED)
else:
    videos = [VIDEO_SOURCE] if VIDEO_SOURCE.is_file() else []

assert videos, f'No supported videos found at {VIDEO_SOURCE}'
if OUTPUT_VIDEO is not None and len(videos) != 1:
    raise ValueError('OUTPUT_VIDEO can only be used with one input video.')

command = [
    sys.executable, str(PROJECT_ROOT / 'run_pipeline.py'),
    '--config', str(PROJECT_ROOT / 'config/config.yaml'),
    '--input', str(VIDEO_SOURCE),
    '--output-dir', str(OUTPUT_ROOT),
]
if OUTPUT_VIDEO is not None:
    command.extend(['--output', str(OUTPUT_VIDEO)])
if RECURSIVE:
    command.append('--recursive')
if STOP_ON_ERROR:
    command.append('--stop-on-error')
if DETECTOR_WEIGHTS is not None:
    command.extend(['--detector-weights', str(DETECTOR_WEIGHTS)])
if CLASSIFIER_WEIGHTS is not None:
    command.extend(['--classifier-weights', str(CLASSIFIER_WEIGHTS)])
if DEVICE is not None:
    command.extend(['--device', DEVICE])
if MAX_FRAMES is not None:
    command.extend(['--max-frames', str(MAX_FRAMES)])

print('Videos:', len(videos))
print('Command:', ' '.join(command))

In [ ]:
completed = subprocess.run(command, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
completed.check_returncode()

## Results

Review the batch summary and preview every annotated output.

In [ ]:
batch_path = OUTPUT_ROOT / 'batch_summary.json'
batch = json.loads(batch_path.read_text(encoding='utf-8'))
assert batch['input_count'] == len(videos)
assert batch['failed_videos'] == 0, batch['results']

print('Successful:', batch['successful_videos'])
print('Failed:', batch['failed_videos'])
for item in batch['results']:
    print(Path(item['input_video']).name, item['status'])
    if item['status'] == 'succeeded':
        print('  Summary:', item['summary'])
        print('  Processing FPS:', item['benchmark']['processing_fps'])

In [ ]:
from IPython.display import Video, display

for annotated_path in sorted(OUTPUT_ROOT.glob('*/annotated.mp4')):
    print(annotated_path.parent.name)
    display(Video(str(annotated_path), embed=True, width=720))

## Next Steps

Download the complete output directory for inspection or archival.

In [ ]:
from google.colab import files

archive_path = shutil.make_archive(
    '/content/cherry-tomato-vision-sorting-outputs',
    'zip',
    OUTPUT_ROOT,
)
files.download(archive_path)